# **Import Essential Packages**
**Name**: B01_zero_shot.ipynb

**Purpose**: In this notebook, I build a simple baseline that integretates LLM power as the Head of the framework to understanding questions an generate answers.

**Author**: Khang Phuc Nguyen

**Date**: 09-05-2026

In [1]:
!python -m pip install --upgrade pip
!pip install -q pandas tqdm

  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1


# **1. Essential Imports and Configuration**

This section prepares everything that the zero-shot baseline needs before we call any model:

- locate the project root reliably;
- define paths to the organizer datasets;
- define where predictions/reports will be saved;
- configure reproducibility and model settings;
- keep the notebook easy to run in both **mock mode** and **real local LLM mode**.

For the first baseline, we want the setup to be simple and explicit because later experiments will compare against this zero-shot result.

In [2]:
# Python standard library imports.
# These modules are enough for path handling, JSON files, environment variables,
# reproducibility, and simple timestamped artifact names.
import json
import os
import random
import sys
from datetime import datetime
from pathlib import Path

# Third-party imports.
# pandas is used for tabular dataset handling.
# tqdm gives progress bars when we later run the model over many examples.
import pandas as pd
from tqdm.auto import tqdm


# -----------------------------------------------------------------------------
# 1. Locate the repository root
# -----------------------------------------------------------------------------
# A notebook can be launched from different working directories depending on
# whether you open it from VS Code, JupyterLab, or the terminal. Instead of
# assuming the current directory is the project root, we walk upward until we
# find pyproject.toml, which identifies the Exact2026 repository.
def find_repo_root(start: Path | None = None) -> Path:
    """Return the Exact2026 repository root by searching for pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        "Could not find pyproject.toml. Please run this notebook inside the Exact2026 repository."
    )


ROOT = find_repo_root()
SRC_DIR = ROOT / "src"

# Add src/ to Python's import path so future project modules can be imported as
# normal packages. This is useful once we implement reusable code outside the
# notebook, for example src/baselines/llm_client.py or evaluation utilities.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# -----------------------------------------------------------------------------
# 2. Dataset paths
# -----------------------------------------------------------------------------
# The organizer released two text-only datasets:
# - Type 1: logic / education regulation questions with natural-language premises.
# - Type 2: physics questions with CoT, answer, and unit fields.
DATA_DIR = SRC_DIR / "data"
TYPE1_PATH = (
    DATA_DIR
    / "Logic_Based_Educational_Queries_Text_Only"
    / "Logic_Based_Educational_Queries.json"
)
TYPE2_PATH = DATA_DIR / "Physics_Problems_Text_Only.csv"

# Check early that the expected files exist. Failing here is easier to debug than
# failing later inside a loading/evaluation loop.
assert TYPE1_PATH.is_file(), f"Type 1 dataset not found: {TYPE1_PATH}"
assert TYPE2_PATH.is_file(), f"Type 2 dataset not found: {TYPE2_PATH}"


# -----------------------------------------------------------------------------
# 3. Artifact paths
# -----------------------------------------------------------------------------
# All outputs from this baseline should go to artifacts/ so that:
# - notebooks stay clean;
# - predictions can be inspected later;
# - metrics can be reused in the paper/report;
# - future baselines can be compared fairly.
ARTIFACTS_DIR = ROOT / "artifacts"
PREDICTIONS_DIR = ARTIFACTS_DIR / "predictions"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
SPLITS_DIR = ARTIFACTS_DIR / "splits"

for directory in [ARTIFACTS_DIR, PREDICTIONS_DIR, REPORTS_DIR, SPLITS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# A run name makes saved files easy to identify. We keep it deterministic enough
# to read but unique enough that multiple experiments do not overwrite each other.
RUN_NAME = "B01_zero_shot"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = f"{RUN_NAME}_{RUN_TIMESTAMP}"


# -----------------------------------------------------------------------------
# 4. Reproducibility settings
# -----------------------------------------------------------------------------
# Zero-shot LLM outputs may still vary depending on sampling settings. We set a
# seed now so that any random splits, sampling, or mock outputs are repeatable.
SEED = 42
random.seed(SEED)

# Optional split file. If it does not exist yet, later cells can create it or run
# on the full dataset. Keeping the path here makes future evaluation consistent.
SPLIT_FILE = SPLITS_DIR / f"seed{SEED}.json"


# -----------------------------------------------------------------------------
# 5. Model and inference settings
# -----------------------------------------------------------------------------
# The challenge restricts submitted systems to open-source/open-weight LLMs with
# at most 8B parameters. For this zero-shot baseline, we keep model selection in
# environment variables so the same notebook can run with different local models.
#
# Example terminal usage:
#   EXACT_MODEL_ID="Qwen/Qwen2.5-7B-Instruct" jupyter notebook
#   MOCK_LLM=1 jupyter notebook
MODEL_ID = os.getenv("EXACT_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")

# MOCK_LLM=1 lets us test data loading, prompt building, parsing, and evaluation
# without downloading or running a real model. This is useful for quick debugging.
MOCK_LLM = os.getenv("MOCK_LLM", "0") == "1"

# Conservative generation settings for a baseline. Low temperature reduces
# randomness, which makes evaluation and error analysis easier.
MAX_NEW_TOKENS = int(os.getenv("EXACT_MAX_NEW_TOKENS", "512"))
TEMPERATURE = float(os.getenv("EXACT_TEMPERATURE", "0.0"))
TOP_P = float(os.getenv("EXACT_TOP_P", "1.0"))


# -----------------------------------------------------------------------------
# 6. Small preview of the configuration
# -----------------------------------------------------------------------------
print("Exact2026 zero-shot baseline configuration")
print(f"  Repository root : {ROOT}")
print(f"  Type 1 dataset  : {TYPE1_PATH}")
print(f"  Type 2 dataset  : {TYPE2_PATH}")
print(f"  Artifacts dir   : {ARTIFACTS_DIR}")
print(f"  Run ID          : {RUN_ID}")
print(f"  Seed            : {SEED}")
print(f"  Model ID        : {MODEL_ID}")
print(f"  Mock LLM        : {MOCK_LLM}")
print(f"  Max new tokens  : {MAX_NEW_TOKENS}")
print(f"  Temperature     : {TEMPERATURE}")
print(f"  Top-p           : {TOP_P}")

Exact2026 zero-shot baseline configuration
  Repository root : /home/phuckhang/MyWorkspace/Exact2026
  Type 1 dataset  : /home/phuckhang/MyWorkspace/Exact2026/src/data/Logic_Based_Educational_Queries_Text_Only/Logic_Based_Educational_Queries.json
  Type 2 dataset  : /home/phuckhang/MyWorkspace/Exact2026/src/data/Physics_Problems_Text_Only.csv
  Artifacts dir   : /home/phuckhang/MyWorkspace/Exact2026/artifacts
  Run ID          : B01_zero_shot_20260512_110636
  Seed            : 42
  Model ID        : Qwen/Qwen2.5-7B-Instruct
  Mock LLM        : False
  Max new tokens  : 512
  Temperature     : 0.0
  Top-p           : 1.0


/home/phuckhang/MyWorkspace/Exact2026/exact/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **2. Data Loading and Normalization**

The organizer released two datasets with different structures:

1. **Type 1 / Logic** is a JSON list. Each item contains one premise set and one or more questions about that same premise set.
2. **Type 2 / Physics** is a CSV file. Each row is one physics problem.

For the zero-shot baseline, we normalize both datasets into one row-per-question table. This makes later prompting and evaluation much easier because every row has the same core fields:

- `id`: unique example id;
- `task_type`: `logic` or `physics`;
- `question`: the question to answer;
- `gold_answer`: organizer-provided answer;
- `gold_explanation`: organizer explanation or CoT;
- task-specific metadata such as premises, FOL, unit, and ID prefix.

In [ ]:
def detect_logic_question_type(question: str, answer: str) -> str:
    """Classify Type 1 questions into broad answer-format groups."""
    question = question or ""
    answer = str(answer or "").strip()

    # Multiple-choice questions in this dataset usually include options A-D inside
    # the question text. Their gold answer is normally one of A, B, C, or D.
    if "\nA." in question or answer in {"A", "B", "C", "D"}:
        return "multiple_choice"

    # The official task commonly uses Yes / No / Unknown labels for entailment.
    if answer in {"Yes", "No", "Unknown", "Uncertain", "False"}:
        return "yes_no_unknown"

    return "other"


def detect_physics_answer_type(answer: str, unit: str) -> str:
    """Classify Type 2 answers by whether they look numeric, conceptual, or missing."""
    answer = str(answer or "").strip()
    unit = str(unit or "").strip()

    if not answer:
        return "missing_answer"

    # This intentionally stays simple. Later we can improve numeric parsing for
    # fractions, scientific notation, intervals, and vector answers.
    has_digit = any(ch.isdigit() for ch in answer)
    if has_digit:
        return "numeric_with_unit" if unit and unit not in {"-", "—"} else "numeric_no_unit"

    return "conceptual"


def physics_id_prefix(example_id: str) -> str:
    """Return the alphabetic prefix of a physics ID, e.g. TD401 -> TD."""
    prefix = ""
    for ch in str(example_id):
        if ch.isalpha():
            prefix += ch
        else:
            break
    return prefix or "unknown"


def load_logic_dataset(path: Path) -> pd.DataFrame:
    """Load and flatten the Type 1 logic dataset into one row per question."""
    raw_records = json.loads(path.read_text(encoding="utf-8"))
    rows = []

    for group_index, record in enumerate(raw_records):
        premises_nl = record.get("premises-NL", [])
        premises_fol = record.get("premises-FOL", [])
        support_indices = record.get("idx", [])
        questions = record.get("questions", [])
        answers = record.get("answers", [])
        explanations = record.get("explanation", [])

        # Each original record can contain multiple questions over the same
        # premise set. We create one normalized row per question.
        for question_index, question in enumerate(questions):
            answer = answers[question_index] if question_index < len(answers) else ""
            explanation = explanations[question_index] if question_index < len(explanations) else ""
            support_idx = support_indices[question_index] if question_index < len(support_indices) else []
            question_type = detect_logic_question_type(question, answer)

            rows.append(
                {
                    "id": f"logic_{group_index:04d}_{question_index:02d}",
                    "group_id": f"logic_{group_index:04d}",
                    "task_type": "logic",
                    "question_type": question_type,
                    "question": question,
                    "premises_nl": premises_nl,
                    "premises_fol": premises_fol,
                    "support_idx": support_idx,
                    "gold_answer": str(answer).strip(),
                    "gold_unit": "",
                    "gold_explanation": explanation,
                    "source_path": str(path),
                    # This label is used only for split balancing.
                    "stratify_label": f"logic::{question_type}",
                }
            )

    return pd.DataFrame(rows)


def load_physics_dataset(path: Path) -> pd.DataFrame:
    """Load the Type 2 physics CSV dataset into the normalized row format."""
    raw_df = pd.read_csv(path).fillna("")
    rows = []

    for _, record in raw_df.iterrows():
        example_id = str(record.get("id", "")).strip()
        answer = str(record.get("answer", "")).strip()
        unit = str(record.get("unit", "")).strip()
        prefix = physics_id_prefix(example_id)
        answer_type = detect_physics_answer_type(answer, unit)

        rows.append(
            {
                "id": f"physics_{example_id}",
                "group_id": f"physics_{example_id}",
                "task_type": "physics",
                "question_type": "physics",
                "question": str(record.get("question", "")).strip(),
                "premises_nl": [],
                "premises_fol": [],
                "support_idx": [],
                "gold_answer": answer,
                "gold_unit": unit,
                "gold_explanation": str(record.get("cot", "")).strip(),
                "source_path": str(path),
                "id_prefix": prefix,
                "answer_type": answer_type,
                # This label balances both physics topic family and answer style.
                "stratify_label": f"physics::{prefix}::{answer_type}",
            }
        )

    return pd.DataFrame(rows)


logic_df = load_logic_dataset(TYPE1_PATH)
physics_df = load_physics_dataset(TYPE2_PATH)

print("Loaded normalized datasets")
print(f"  Logic rows   : {len(logic_df):,}")
print(f"  Physics rows : {len(physics_df):,}")
print(f"  Total rows   : {len(logic_df) + len(physics_df):,}")
print()
print("Logic question types:")
display(logic_df["question_type"].value_counts().rename_axis("question_type").to_frame("count"))
print("Physics answer types:")
display(physics_df["answer_type"].value_counts().rename_axis("answer_type").to_frame("count"))
print("Physics ID prefixes:")
display(physics_df["id_prefix"].value_counts().rename_axis("id_prefix").to_frame("count"))

# Preview a few normalized examples so we can manually verify that fields look
# correct before building prompts and calling the LLM.
display(logic_df.head(2))
display(physics_df.head(2))

# **3. Train / Dev / Test Split**

For this project, use **70% train / 15% dev / 15% test**.

Why this is the best default here:

- **Train 70%**: enough examples for prompt design, formula-template mining, error taxonomy, and later LoRA/fine-tuning if needed.
- **Dev 15%**: enough examples for fast iteration while building the zero-shot, symbolic, and neuro-symbolic pipelines.
- **Test 15%**: enough examples for a final honest estimate without wasting too much limited challenge data.

How the split affects pipeline performance:

- If the **dev set is too small**, you may tune prompts/templates to noise and make bad decisions.
- If the **test set is used during development**, your reported result becomes optimistic and weak for the paper.
- If the **train set is too small**, later fine-tuning or template mining may miss important physics and logic patterns.
- If Type 1 is split by individual questions instead of premise groups, the same premise set can appear in both train and test. This causes leakage and makes the logic pipeline look stronger than it really is.
- For Type 2, stratifying by ID prefix helps keep physics topic families distributed across splits, so one split does not accidentally contain most of one problem type.

Rule for this notebook:

> Use the **dev split** while improving prompts and code. Touch the **test split** only when reporting the baseline result.

In [ ]:
# ----------------------------------------------------------------------------
# Create reproducible train / dev / test splits
# ----------------------------------------------------------------------------
# Recommended ratio for this project: 70% train, 15% dev, 15% test.
#
# Why this ratio?
# - 70% train keeps enough examples for prompt design, template mining, and later
#   fine-tuning experiments.
# - 15% dev is large enough for fast iteration while building the pipeline.
# - 15% test is kept untouched until we want an honest estimate of performance.
#
# Important: for Type 1, multiple questions can share the same premise set. If we
# split individual questions randomly, the model/pipeline may see the same
# premises in train and test, which causes leakage. Therefore we split Type 1 by
# `group_id`, where each group is one original premise set.
#
# For Type 2, each row is an independent physics problem. We still stratify by
# `stratify_label` so that formula/topic-like ID families are distributed across
# train/dev/test.
SPLIT_RATIOS = {
    "train": 0.70,
    "dev": 0.15,
    "test": 0.15,
}


def assign_group_splits(
    df: pd.DataFrame,
    group_col: str,
    stratify_col: str,
    ratios: dict[str, float],
    seed: int,
) -> pd.DataFrame:
    """Assign train/dev/test labels while keeping every group in only one split."""
    assert abs(sum(ratios.values()) - 1.0) < 1e-9, "Split ratios must sum to 1.0"

    rng = random.Random(seed)
    split_by_group: dict[str, str] = {}

    # Work at group level, not row level. This prevents Type 1 premise leakage.
    group_table = (
        df[[group_col, stratify_col]]
        .drop_duplicates(subset=[group_col])
        .reset_index(drop=True)
    )

    # Stratify groups by task-specific labels. This keeps answer/topic distribution
    # more stable across train/dev/test than pure random splitting.
    for _, label_groups in group_table.groupby(stratify_col):
        groups = label_groups[group_col].tolist()
        rng.shuffle(groups)

        n = len(groups)
        n_train = int(round(n * ratios["train"]))
        n_dev = int(round(n * ratios["dev"]))

        # Ensure all groups are assigned even when rounding is imperfect.
        train_groups = groups[:n_train]
        dev_groups = groups[n_train : n_train + n_dev]
        test_groups = groups[n_train + n_dev :]

        for group in train_groups:
            split_by_group[group] = "train"
        for group in dev_groups:
            split_by_group[group] = "dev"
        for group in test_groups:
            split_by_group[group] = "test"

    output = df.copy()
    output["split"] = output[group_col].map(split_by_group)
    return output


# Logic split: grouped by original premise set to prevent leakage.
logic_df = assign_group_splits(
    logic_df,
    group_col="group_id",
    stratify_col="stratify_label",
    ratios=SPLIT_RATIOS,
    seed=SEED,
)

# Physics split: each physics row is independent, so group_id is just its own ID.
physics_df = assign_group_splits(
    physics_df,
    group_col="group_id",
    stratify_col="stratify_label",
    ratios=SPLIT_RATIOS,
    seed=SEED,
)

# Combine the two datasets after splitting. Keeping a single table makes later
# zero-shot prompting/evaluation easier, while `task_type` still lets us separate
# logic and physics metrics.
all_df = pd.concat([logic_df, physics_df], ignore_index=True)

# Save the normalized dataset and split file so every future baseline uses the
# same examples. This is important for fair comparisons in the paper.
NORMALIZED_DATA_PATH = ARTIFACTS_DIR / "normalized_dataset.jsonl"
SPLIT_FILE = SPLITS_DIR / f"seed{SEED}_grouped_70_15_15.json"

all_df.to_json(NORMALIZED_DATA_PATH, orient="records", lines=True, force_ascii=False)

split_payload = {
    "seed": SEED,
    "ratios": SPLIT_RATIOS,
    "notes": [
        "Type 1 is split by original premise-set group_id to avoid premise leakage.",
        "Type 2 is stratified by ID prefix / answer type label.",
        "Test split should remain untouched until final baseline reporting.",
    ],
    "ids": {
        split_name: all_df.loc[all_df["split"] == split_name, "id"].tolist()
        for split_name in ["train", "dev", "test"]
    },
}
SPLIT_FILE.write_text(json.dumps(split_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Saved normalized dataset to: {NORMALIZED_DATA_PATH}")
print(f"Saved split file to: {SPLIT_FILE}")
print()
print("Overall split counts:")
display(pd.crosstab(all_df["task_type"], all_df["split"], margins=True))
print("\nLogic split by question type:")
display(pd.crosstab(logic_df["question_type"], logic_df["split"], margins=True))
print("\nPhysics split by answer type:")
display(pd.crosstab(physics_df["answer_type"], physics_df["split"], margins=True))
print("\nPhysics split by ID prefix:")
display(pd.crosstab(physics_df["id_prefix"], physics_df["split"], margins=True))